In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q monai nibabel
import os, glob, tarfile, zipfile, json, shutil, gc as pygc
import numpy as np, pandas as pd, torch, torch.nn.functional as F, nibabel as nib
from pathlib import Path
from monai import transforms
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR
device = torch.device("cuda")

# ===== 1. Unpack the same 100 test patients =====
TAR = glob.glob("/kaggle/input/**/BraTS2021_Training_Data.tar", recursive=True)[0]
OUT = "/kaggle/working/test_data"
os.makedirs(OUT, exist_ok=True)
order = {}
with tarfile.open(TAR, "r|") as t:
    for m in t:
        if not m.isfile():
            continue
        case = os.path.basename(os.path.dirname(m.name))
        if case not in order:
            order[case] = len(order)
            if order[case] >= 350:
                break
        if 250 <= order[case] < 350:
            t.extract(m, OUT, filter="data")
MODS = ["flair", "t1ce", "t1", "t2"]
test_files = []
for d in sorted(Path(OUT).glob("**/BraTS2021_*")):
    if d.is_dir():
        imgs = [str(d / f"{d.name}_{m}.nii.gz") for m in MODS]
        seg = d / f"{d.name}_seg.nii.gz"
        if all(os.path.exists(p) for p in imgs) and seg.exists():
            test_files.append({"image": imgs, "label": str(seg), "id": d.name})
print(len(test_files), "test patients", flush=True)

# ===== 2. Find files from the crashed run =====
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: add the train-and-test notebook (crashed version) as input."
    return hits[0]
NORMAL_PTH = find("normal_model.pth")
DICE_CSV = find("dice_all_patients.csv")
UNC_CSV = find("uncertainty_all_patients.csv")
HIST = find("history.json")
dice_df = pd.read_csv(DICE_CSV)
print("Normal model:", NORMAL_PTH, flush=True)

# ===== 3. Load models (kept on CPU, moved to GPU only when used) =====
def load(path):
    m = SwinUNETR(in_channels=4, out_channels=3, feature_size=48)
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    m.load_state_dict(sd)
    for q in m.parameters():
        q.requires_grad_(False)
    return m.eval()

def rebuild_pth(folder, out_path):
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_STORED) as z:
        for root, _, files in os.walk(folder):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, "archive/" + os.path.relpath(full, folder))
    return out_path

def scan(folder, cands):
    for root, _, fs in os.walk(folder, followlinks=True):
        for fn in fs:
            p = os.path.join(root, fn)
            if fn.endswith(".pth") and fn != "normal_model.pth":
                cands.append(p)
            elif fn == "data.pkl":
                cands.append(rebuild_pth(root, f"/kaggle/working/rebuilt_{len(cands)}.pth"))
            elif fn.endswith(".zip") and os.path.getsize(p) < 2e9:
                with zipfile.ZipFile(p) as z:
                    names = z.namelist()
                    if any(n.endswith("data.pkl") for n in names) and not any(n.endswith(".pth") for n in names):
                        cands.append(p)
                    else:
                        dest = f"/kaggle/working/unzipped_{len(cands)}"
                        z.extractall(dest); scan(dest, cands)

tf = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"], ensure_channel_first=True),
    transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])
metric = DiceMetric(include_background=True, reduction="none")

def logits(m, x):
    m.to(device)
    with torch.no_grad(), torch.autocast("cuda"):
        return sliding_window_inference(x, (96, 96, 96), 2, m, overlap=0.5).float()

model = load(NORMAL_PTH)
_d = tf(test_files[0])
_x = torch.as_tensor(np.asarray(_d["image"]))[None].float().to(device); _x[:, 1] = 0
_y = torch.as_tensor(np.asarray(_d["label"]))[None].float().to(device)
cands = []; scan("/kaggle/input", cands)
model_drop, best_et = None, -1
for p in cands:
    try:
        m = load(p)
    except Exception:
        continue
    e = metric(y_pred=(torch.sigmoid(logits(m, _x)) > 0.5).float(), y=_y)[0, 2].item()
    m.cpu(); torch.cuda.empty_cache()
    print(f"ET without T1ce = {e:.2f} -> {p}", flush=True)
    if e > best_et:
        model_drop, best_et = m, e
assert model_drop is not None, "Dropout model not found: add the 'project dropping out' notebook (Version 3)."
assert best_et > 0.2, f"The model found has ET {best_et:.2f} without T1ce, so it is NOT the dropout model. Add 'project dropping out' Version 3 as input."
model.cpu(); del _x, _y; torch.cuda.empty_cache()

# ===== 3b. Redo the dropout rows of Table 1 with the REAL dropout model =====
KEEP = {"all4": [0, 1, 2, 3], "no_FLAIR": [1, 2, 3], "no_T1ce": [0, 2, 3]}
drows = []
for i, f in enumerate(test_files):
    d = tf(f)
    x = torch.as_tensor(np.asarray(d["image"]))[None].float().to(device)
    y = torch.as_tensor(np.asarray(d["label"]))[None].float().to(device)
    for c, keep in KEEP.items():
        xin = torch.zeros_like(x); xin[:, keep] = x[:, keep]
        tc, wt, et = metric(y_pred=(torch.sigmoid(logits(model_drop, xin)) > 0.5).float(), y=y)[0].tolist()
        drows.append({"case": f["id"], "model": "dropout", "condition": c, "TC": tc, "WT": wt, "ET": et})
    del x, y; torch.cuda.empty_cache()
    if (i + 1) % 10 == 0:
        print(f"Dropout Dice: {i+1}/100", flush=True)
model_drop.cpu(); torch.cuda.empty_cache()
dice_df = pd.concat([dice_df[dice_df.model == "normal"], pd.DataFrame(drows)], ignore_index=True)
dice_df.to_csv("/kaggle/working/dice_all_patients.csv", index=False)
print("\nTABLE 1 (corrected): Mean Dice, 100 patients")
print(dice_df.groupby(["model", "condition"])[["TC", "WT", "ET"]].mean().round(3), flush=True)

# ===== 4. Memory-safe Grad-CAM =====
def gradcam_window(m, x, ch):
    store = {}
    h1 = m.decoder2.register_forward_hook(lambda mod, i, o: store.__setitem__("a", o))
    h2 = m.decoder2.register_full_backward_hook(lambda mod, gi, go: store.__setitem__("g", go[0]))
    x = x.clone().requires_grad_(True)
    with torch.enable_grad(), torch.autocast("cuda"):
        out = m(x)
        o = out[:, ch].float()
        mask = (torch.sigmoid(o) > 0.5).float()
        score = (o * mask).sum() if mask.sum() > 0 else o.sum()
    score.backward()
    h1.remove(); h2.remove()
    a, g = store["a"].float().detach(), store["g"].float().detach()
    w = g.mean(dim=(2, 3, 4), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[2:], mode="trilinear", align_corners=False)[0, 0].cpu().numpy()
    del store, out, o, score, mask, a, g, w, x
    torch.cuda.empty_cache()
    return cam

def gradcam_tumor(m, vol, wt_mask, ch, S=96):
    m.to(device)
    full = np.zeros(vol.shape[1:], np.float32)
    if not wt_mask.any():
        return full
    lo = np.maximum(np.argwhere(wt_mask).min(0) - 8, 0)
    hi = np.argwhere(wt_mask).max(0) + 8
    starts = []
    for a, b, n in zip(lo, hi, vol.shape[1:]):
        s = list(range(int(a), int(max(a + 1, b - S + 1)), S)) + [int(max(a, b - S))]
        starts.append(sorted({max(0, min(v, n - S)) for v in s}))
    for x0 in starts[0]:
        for y0 in starts[1]:
            for z0 in starts[2]:
                x = torch.from_numpy(vol[:, x0:x0+S, y0:y0+S, z0:z0+S].copy())[None].float().to(device)
                region = full[x0:x0+S, y0:y0+S, z0:z0+S]
                np.maximum(region, gradcam_window(m, x, ch), out=region)
    return full / (full.max() + 1e-8)

# ===== 5. Grad-CAM averages, 100 patients (saves progress every 10) =====
names, gc_rows = {0: "TC", 1: "WT", 2: "ET"}, []
for i, f in enumerate(test_files):
    d = tf(f)
    vol, lab = np.asarray(d["image"]), np.asarray(d["label"]).astype(bool)
    if lab[1].any():
        for mname, m, other in [("normal", model, model_drop), ("dropout", model_drop, model)]:
            other.cpu(); torch.cuda.empty_cache()
            for c in ["all4", "no_T1ce"]:
                v = vol.copy()
                if c == "no_T1ce":
                    v[1] = 0
                for ch in [0, 1, 2]:
                    if lab[ch].any():
                        cam = gradcam_tumor(m, v, lab[1], ch)
                        gc_rows.append({"case": f["id"], "model": mname, "condition": c, "region": names[ch],
                                        "inside": cam[lab[ch]].sum() / (cam.sum() + 1e-8)})
    pygc.collect()
    if (i + 1) % 10 == 0:
        pd.DataFrame(gc_rows).to_csv("/kaggle/working/gradcam_all_patients.csv", index=False)
        print(f"Grad-CAM: {i+1}/100", flush=True)
gcdf = pd.DataFrame(gc_rows)
gcdf.to_csv("/kaggle/working/gradcam_all_patients.csv", index=False)
print("\nTABLE 3: Attention inside the real region (%)")
print((gcdf.pivot_table(index=["model", "condition"], columns="region", values="inside") * 100).round(0), flush=True)
model_drop.cpu(); torch.cuda.empty_cache()

# ===== 6. 20 website patients =====
WEB = Path("/kaggle/working/website_patients"); WEB.mkdir(exist_ok=True)
n_all = dice_df[(dice_df.model == "normal") & (dice_df.condition == "all4")].copy()
n_all["mean"] = n_all[["TC", "WT", "ET"]].mean(axis=1, skipna=True)
n_all = n_all.sort_values("mean", ascending=False).reset_index(drop=True)
picks = n_all.iloc[np.linspace(0, len(n_all) - 1, 20).round().astype(int)]
by_id = {f["id"]: f for f in test_files}

def to_labels(pred):
    out = np.zeros(pred.shape[1:], np.uint8)
    out[pred[1]] = 2; out[pred[0]] = 1; out[pred[2]] = 4
    return out

def location(mask, brain, affine):
    codes = nib.aff2axcodes(affine)
    c, mid = np.argwhere(mask).mean(0), np.argwhere(brain).mean(0)
    words = {"L": ("left", "right"), "R": ("right", "left"), "A": ("front", "back"),
             "P": ("back", "front"), "S": ("upper", "lower"), "I": ("lower", "upper")}
    return ", ".join(words[code][0] if c[k] > mid[k] else words[code][1] for k, code in enumerate(codes))

report = []
for rank, (_, row) in enumerate(picks.iterrows()):
    f = by_id[row["case"]]
    d = tf(f)
    vol, lab = np.asarray(d["image"]), np.asarray(d["label"]).astype(bool)
    x = torch.from_numpy(vol)[None].float().to(device)
    pred = (torch.sigmoid(logits(model, x)) > 0.5)[0].cpu().numpy()
    xn = x.clone(); xn[:, 1] = 0
    pred_no = (torch.sigmoid(logits(model, xn)) > 0.5)[0].cpu().numpy()
    del x, xn; torch.cuda.empty_cache()
    img = nib.load(f["image"][0]); aff, hdr = img.affine, img.header
    pdir = WEB / f["id"]; pdir.mkdir(exist_ok=True)
    shutil.copy(f["image"][0], pdir / "flair.nii.gz")
    shutil.copy(f["label"], pdir / "truth.nii.gz")
    nib.save(nib.Nifti1Image(to_labels(pred), aff), pdir / "prediction.nii.gz")
    nib.save(nib.Nifti1Image(to_labels(pred_no), aff), pdir / "prediction_no_t1ce.nii.gz")
    for name, ch in [("wt", 1), ("tc", 0), ("et", 2)]:
        cam = gradcam_tumor(model, vol, pred[1] if pred[1].any() else lab[1], ch)
        nib.save(nib.Nifti1Image((cam * 255).astype(np.uint8), aff), pdir / f"gradcam_{name}.nii.gz")
    vox = float(np.prod(hdr.get_zooms()[:3])) / 1000
    no_row = dice_df[(dice_df.case == f["id"]) & (dice_df.model == "normal") & (dice_df.condition == "no_T1ce")].iloc[0]
    kind = "best" if rank < 4 else ("hard" if rank >= 16 else "typical")
    report.append({
        "id": f["id"], "case_type": kind, "tumor_type": "Glioma (all BraTS 2021 patients)",
        "dice": {k: round(float(row[k]), 3) for k in ["WT", "TC", "ET"]},
        "dice_without_t1ce": {k: round(float(no_row[k]), 3) for k in ["WT", "TC", "ET"]},
        "volume_cm3": {"WT": round(float(pred[1].sum() * vox), 1), "TC": round(float(pred[0].sum() * vox), 1),
                       "ET": round(float(pred[2].sum() * vox), 1)},
        "location_approx": location(lab[1] if lab[1].any() else pred[1], vol[0] != 0, aff),
    })
    pygc.collect()
    print(f"Website patient {rank+1}/20: {f['id']} ({kind})", flush=True)
json.dump(report, open(WEB / "patients.json", "w"), indent=1)

# ===== 7. Pack everything =====
for p in [NORMAL_PTH, UNC_CSV, HIST]:
    shutil.copy(p, "/kaggle/working/" + os.path.basename(p))
shutil.rmtree(OUT, ignore_errors=True)
!cd /kaggle/working && zip -qr final_results.zip normal_model.pth history.json dice_all_patients.csv uncertainty_all_patients.csv gradcam_all_patients.csv website_patients && ls -lh final_results.zip
print("ALL DONE: download final_results.zip from Output")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.7 MB/s eta 0:00:00
100 test patients
Normal model: /kaggle/input/notebooks/sathyaaaa/new-test-00/normal_model.pth


nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(


ET without T1ce = 0.53 -> /kaggle/input/notebooks/sathyaaaa/project-dropping-out/best_swinunetr.pth
ET without T1ce = 0.01 -> /kaggle/input/notebooks/sathyaaaa/new-test-00/rebuilt_0.pth
Dropout Dice: 10/100
Dropout Dice: 20/100
Dropout Dice: 30/100
Dropout Dice: 40/100
Dropout Dice: 50/100
Dropout Dice: 60/100
Dropout Dice: 70/100
Dropout Dice: 80/100
Dropout Dice: 90/100
Dropout Dice: 100/100

TABLE 1 (corrected): Mean Dice, 100 patients
                      TC     WT     ET
model   condition                     
dropout all4       0.837  0.911  0.855
        no_FLAIR   0.805  0.831  0.828
        no_T1ce    0.563  0.892  0.391
normal  all4       0.880  0.915  0.860
        no_FLAIR   0.571  0.434  0.581
        no_T1      0.856  0.898  0.836
        no_T1ce    0.199  0.870  0.015
        no_T2      0.847  0.885  0.853
Grad-CAM: 10/100
Grad-CAM: 20/100
Grad-CAM: 30/100
Grad-CAM: 40/100
Grad-CAM: 50/100
Grad-CAM: 60/100
Grad-CAM: 70/100
Grad-CAM: 80/100
Grad-CAM: 90/100
Grad-CAM: 100/